## Apartments scraping — Bezrealitky.cz

Scrapes rental listings from bezrealitky.cz for Prague.
First collects listing URIs from search result pages, then fetches JSON data for each listing.
Results saved to `data/raw/apartments.csv`.

Note: Bezrealitky uses a Next.js build hash in their JSON API URLs. This hash changes
with each site deployment, so the scraper detects it automatically from the page source.

In [ ]:
import requests
import pandas as pd
import time
import re
import os
from bs4 import BeautifulSoup

headers = {'User-Agent': 'JEM207 StudentProject (Educational use; contact: your_email@fsv.cuni.cz)'}

BASE_URL = (
    'https://www.bezrealitky.cz/vyhledat'
    '?currency=CZK&estateType=BYT&offerType=PRONAJEM'
    '&osm_value=Praha%2C+%C4%8Cesko&regionOsmIds=R435541&location=exact'
)

In [ ]:
# detect current build hash from the page source
r = requests.get(BASE_URL, headers=headers)
m = re.search(r'/_next/data/([a-f0-9]+)/cs/', r.text)
if m:
    build_hash = m.group(1)
    print(f'build hash: {build_hash}')
else:
    raise RuntimeError('Could not find build hash — site structure may have changed')

In [ ]:
# collect listing URIs across all pages
all_uris = set()

for page in range(1, 93):
    url = f'{BASE_URL}&page={page}'
    try:
        r = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(r.text, 'html.parser')
        links = soup.find_all('a', href=True)
        uris = [
            l['href'].split('/nemovitosti-byty-domy/')[1]
            for l in links
            if 'nabidka-pronajem-bytu' in l['href']
        ]
        all_uris.update(uris)
        if page % 10 == 0:
            print(f'page {page} — {len(all_uris)} URIs so far')
    except Exception as e:
        print(f'page {page} failed: {e}')
        time.sleep(5)
        continue
    time.sleep(3)

print(f'total unique URIs: {len(all_uris)}')

In [ ]:
# fetch listing details from JSON API
listings = []
failed = 0

for uri in all_uris:
    json_url = f'https://www.bezrealitky.cz/_next/data/{build_hash}/cs/nemovitosti-byty-domy/{uri}.json'
    try:
        r = requests.get(json_url, headers=headers, timeout=15)
        advert = r.json()['pageProps']['origAdvert']
        listings.append({
            'price':       advert['price'],
            'charges':     advert['charges'],
            'surface':     advert['surface'],
            'disposition': advert['disposition'],
            'address':     advert['address'],
            'district':    advert['regionTree'][-1]['name'],
        })
    except Exception:
        failed += 1
        continue
    time.sleep(2)

print(f'collected {len(listings)} listings ({failed} failed)')

In [ ]:
df = pd.DataFrame(listings)
print(df.head())
print(df.info())

In [ ]:
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/apartments.csv', index=False)
print('saved')